# 流水特征工程 v2.1（严格快照 & 日期化计算）

In [1]:
import os, numpy as np, pandas as pd
SEED=42
np.random.seed(SEED)
SECONDS_PER_DAY=86400.0

def to_dt(series):
    return pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', utc=True)


In [2]:
# ===== 路径配置（请按实际环境修改） =====
TRAIN_CSV = 'train/train.csv'
TRAIN_STMT_CSV = 'train/train_bank_statement.csv'
TEST_CSV = 'testaa/testaa.csv'              # 如果没有可置为 None
TEST_STMT_CSV = 'testaa/testaa_bank_statement.csv'  # 如果没有可置为 None

OUT_TRAIN_FEAT = 'train/train_statement_feature_v2.csv'
OUT_TEST_FEAT  = 'testaa/testaa_statement_feature_v2.csv'

In [3]:
def _coerce_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def _safe_div(a,b):
    b=np.where(b==0,np.nan,b)
    return a/b

def _prep_stmt(stmt: pd.DataFrame) -> pd.DataFrame:
    df=stmt.copy()
    for col in df.columns:
        if df[col].dtype=='O':
            df[col]=df[col].replace(['',' ','nan','NaN','NULL','None'], np.nan)
    df['id']=_coerce_numeric(df['id']).astype('Int64')
    df['amount']=_coerce_numeric(df['amount']).astype(float)
    df['direction']=_coerce_numeric(df['direction']).astype('Int8')
    df['time']=to_dt(df['time'])
    df=df.dropna(subset=['id','time','direction','amount'])
    df['abs_amount']=df['amount'].abs()
    df['is_income']=(df['direction']==0).astype('Int8')
    df['is_expense']=(df['direction']==1).astype('Int8')
    df['date']=df['time'].dt.floor('D')
    return df

def build_statement_features(stmt: pd.DataFrame, base_df: pd.DataFrame, id_col='id', record_col='record_time', half_life_days=60.0) -> pd.DataFrame:
    if stmt is None or len(stmt)==0:
        return pd.DataFrame({id_col: base_df[id_col].unique()})
    anchor=base_df[[id_col,record_col]].drop_duplicates().copy()
    anchor['record_dt']=to_dt(anchor[record_col])
    df=_prep_stmt(stmt).merge(anchor[[id_col,'record_dt']], on=id_col, how='inner')
    df=df[df['time']<=df['record_dt']]
    g=df.groupby(id_col, observed=True)
    basic=g.agg(tx_count=('amount','size'), income_count=('is_income','sum'), expense_count=('is_expense','sum'),
                sum_income=('amount', lambda s: s[df.loc[s.index,'is_income']==1].sum()),
                sum_expense_abs=('amount', lambda s: s[df.loc[s.index,'is_expense']==1].abs().sum()),
                mean_abs_amount=('abs_amount','mean'), std_abs_amount=('abs_amount','std'),
                min_abs_amount=('abs_amount','min'), max_abs_amount=('abs_amount','max'),
                active_days=('date','nunique'), max_time=('time','max'), min_time=('time','min')).reset_index()
    q=df.groupby(id_col)['abs_amount'].quantile([0.5,0.75,0.9]).unstack(); q.columns=['amt_p50','amt_p75','amt_p90']; q=q.reset_index()
    feat=basic.merge(q, on=id_col, how='left')
    delta_days=(df['record_dt']-df['time']).dt.total_seconds()/SECONDS_PER_DAY
    w=np.power(0.5, delta_days/float(half_life_days))
    df['w']=w
    df['w_income_amt']=df['amount'].where(df['is_income']==1,0.0)*df['w']
    df['w_expense_abs']=df['amount'].where(df['is_expense']==1,0.0).abs()*df['w']
    ewm=df.groupby(id_col, observed=True).agg(ewm_income=('w_income_amt','sum'), ewm_expense_abs=('w_expense_abs','sum'), ewm_tx=('w','sum')).reset_index()
    feat=feat.merge(ewm, on=id_col, how='left')
    for c in ['ewm_income','ewm_expense_abs','ewm_tx']:
        feat[c]=feat[c].fillna(0.0)
    feat['ewm_income_expense_ratio']=_safe_div(feat['ewm_income'], feat['ewm_expense_abs'])
    feat=feat.merge(anchor[[id_col,'record_dt']], on=id_col, how='right')
    feat['last_tx_days_before_record']=((feat['record_dt']-feat['max_time']).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    feat['first_tx_days_before_record']=((feat['record_dt']-feat['min_time']).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    feat['span_days']=((feat['max_time']-feat['min_time']).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    feat['has_statement']=np.where(feat['tx_count'].fillna(0)>0,1,0).astype(int)
    feat=feat.drop(columns=['max_time','min_time'])
    num_cols=feat.select_dtypes(include=[np.number]).columns.tolist()
    if 'id' in num_cols: num_cols.remove('id')
    med=feat[num_cols].median()
    feat[num_cols]=feat[num_cols].fillna(med)
    return feat

In [4]:
df_train=pd.read_csv(TRAIN_CSV)
feat_train=build_statement_features(pd.read_csv(TRAIN_STMT_CSV), df_train, 'id','record_time')
feat_train.to_csv(OUT_TRAIN_FEAT, index=False, encoding='utf-8')
print('train_statement_feature_v21.csv 保存：', OUT_TRAIN_FEAT, feat_train.shape)

if TEST_CSV is not None and os.path.exists(TEST_CSV):
    df_test=pd.read_csv(TEST_CSV)
    if TEST_STMT_CSV is not None and os.path.exists(TEST_STMT_CSV):
        feat_test=build_statement_features(pd.read_csv(TEST_STMT_CSV), df_test, 'id','record_time')
    else:
        feat_test=pd.DataFrame({'id': df_test['id'].unique()})
    feat_test.to_csv(OUT_TEST_FEAT, index=False, encoding='utf-8')
    print('testaa_statement_feature_v21.csv 保存：', OUT_TEST_FEAT, feat_test.shape)
else:
    print('未发现测试集，已跳过。')

train_statement_feature_v21.csv 保存： train/train_statement_feature_v2.csv (53480, 23)
testaa_statement_feature_v21.csv 保存： testaa/testaa_statement_feature_v2.csv (20054, 23)


In [5]:
dfb=pd.read_csv(TRAIN_STMT_CSV)
base=pd.read_csv(TRAIN_CSV)[['id','record_time']].drop_duplicates()
dfb=dfb.merge(base, on='id', how='inner')
ratio_future=((to_dt(dfb['time'])>to_dt(dfb['record_time'])).mean())
print('原始流水中 record_time 之后交易占比：', round(float(ratio_future),6))

原始流水中 record_time 之后交易占比： 0.255754
